# Menghitung Jumlah Jerawat dengan YOLO26 — Tahap 1: Deteksi (Bounding Box) + Counting

Notebook ini adalah **titik awal penelitian**: melatih model **YOLO26** (Ultralytics) untuk mendeteksi setiap jerawat
dengan *bounding box*, lalu **menghitung jumlah jerawat** pada setiap gambar wajah (total dan per jenis).

## Alur notebook
| No | Langkah | Tujuan |
|---|---|---|
| 1 | Cek GPU | Memastikan training berjalan di GPU |
| 2 | Instalasi library | Ultralytics (YOLO26) + Roboflow |
| 3 | Konfigurasi | Semua parameter penting di satu tempat |
| 4 | Menyiapkan dataset deteksi | Dataset berformat YOLO (gambar + label bbox) |
| 5 | Merapikan `data.yaml` | Path dataset yang pasti benar |
| 6 | Eksplorasi dataset (EDA) | Memahami jumlah jerawat per gambar & ukuran bbox |
| 7 | Training YOLO26 | Transfer learning dari bobot COCO |
| 8 | Kurva training | Memantau loss & mAP |
| 9 | Evaluasi deteksi | Precision, Recall, mAP50, mAP50-95 |
| 10 | Fungsi penghitung jerawat | Inti dari *counting* |
| 11 | Demo visual | Ground truth vs prediksi |
| 12 | Tuning confidence threshold | Memilih ambang terbaik untuk *counting* (pakai data validasi) |
| 13 | Evaluasi counting | MAE, RMSE, akurasi ±1, dsb. di data test |
| 14 | Hitung jerawat pada gambar baru | Upload gambar / satu folder → CSV |
| 15 | Simpan hasil | Bobot model & tabel hasil ke Google Drive |

## ⚠️ Catatan penting tentang dataset Anda
Dataset `Acne Classification.v12i.folder` dari Roboflow diekspor dalam **format *folder* (klasifikasi)**:
setiap gambar hanya punya **satu label untuk seluruh gambar** (`Cyst`, `Papula`, atau `Pustula`) dan
**tidak memiliki koordinat bounding box**. Padahal untuk menghitung jerawat, model harus tahu **di mana setiap jerawat berada**.

Karena itu, sebelum training Anda perlu **dataset deteksi objek** (format YOLO). Caranya dijelaskan di **Langkah 4**:
1. Upload gambar Anda ke project Roboflow bertipe **Object Detection**, lalu gambar kotak pada **setiap** jerawat; atau
2. Gunakan dataset deteksi jerawat publik dari Roboflow Universe (mis. turunan ACNE04) sebagai awal/pembanding.

Setelah dataset deteksi tersedia, seluruh notebook ini bisa dijalankan dari atas ke bawah tanpa perubahan kode selain konfigurasi.

## 1. Cek GPU
YOLO26 bisa dilatih di CPU, tetapi akan **sangat lambat**. Aktifkan GPU di Colab:
**Runtime → Change runtime type → Hardware accelerator: T4 GPU** → Save.

- `nvidia-smi` menampilkan tipe GPU dan memori yang tersedia.
- `torch.cuda.is_available()` harus bernilai `True`. Jika `False`, training akan memakai CPU.

In [ ]:
!nvidia-smi
import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

## 2. Instalasi library
- **`ultralytics`**: library resmi YOLO. YOLO26 tersedia mulai versi **8.4.x**, sehingga kita pakai `-U` agar mendapat versi terbaru.
- **`roboflow`**: untuk mengunduh dataset dari Roboflow langsung ke Colab.

`ultralytics.checks()` mencetak ringkasan versi, GPU, RAM, dan disk — berguna untuk dicatat di bagian *experimental setup* laporan penelitian.

**Sekilas tentang YOLO26** (relevan untuk bab tinjauan pustaka):
- **End-to-end / NMS-free**: prediksi langsung keluar tanpa *Non-Maximum Suppression*, sehingga inferensi lebih sederhana dan cepat.
- **Tanpa DFL** (*Distribution Focal Loss*) → kepala deteksi lebih ringan dan mudah diekspor (ONNX, TFLite, dsb.).
- **ProgLoss + STAL** (*Small-Target-Aware Label assignment*) → dirancang membantu deteksi **objek kecil**. Ini sangat relevan karena jerawat umumnya objek kecil di gambar wajah.
- **Optimizer MuSGD** (gabungan SGD dan Muon) untuk konvergensi training yang lebih stabil.
- Tersedia ukuran `n` (nano), `s` (small), `m` (medium), `l` (large), `x` (extra large).

In [ ]:
!pip install -q -U ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3. Konfigurasi
Semua parameter yang sering diubah dikumpulkan di sini supaya eksperimen mudah diulang (*reproducible*).

| Parameter | Penjelasan |
|---|---|
| `MODEL_WEIGHTS` | Bobot awal YOLO26 (pretrained COCO). `yolo26n.pt` paling cepat, `yolo26s.pt` titik awal yang seimbang, `m/l/x` lebih akurat tapi lebih berat. |
| `EPOCHS` | Berapa kali model melihat seluruh data training. 100 cukup untuk awal; *early stopping* akan berhenti lebih cepat jika tidak ada perbaikan. |
| `IMGSZ` | Ukuran input model. Gambar Anda 512×512; memakai 640 sedikit memperbesar gambar sehingga jerawat kecil lebih "terlihat" oleh model. |
| `BATCH` | Jumlah gambar per langkah. 16 aman untuk GPU T4. Isi `-1` agar Ultralytics memilih otomatis. |
| `PATIENCE` | *Early stopping*: berhenti bila mAP validasi tidak membaik selama N epoch. |
| `SEED` | Agar hasil bisa diulang. |
| `CONF_DEFAULT` | Ambang *confidence* awal: kotak dengan skor di bawah ini tidak dihitung. Nanti dituning di Langkah 12. |
| `USE_DRIVE` | Jika `True`, semua hasil disimpan ke Google Drive sehingga tidak hilang saat sesi Colab terputus. |

In [ ]:
from pathlib import Path

MODEL_WEIGHTS = "yolo26s.pt"   # pilihan: yolo26n.pt / yolo26s.pt / yolo26m.pt / yolo26l.pt / yolo26x.pt
EPOCHS        = 100
IMGSZ         = 640
BATCH         = 16
PATIENCE      = 30
SEED          = 42
CONF_DEFAULT  = 0.25
USE_DRIVE     = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/penelitian_jerawat_yolo26')
else:
    OUTPUT_DIR = Path('/content/penelitian_jerawat_yolo26')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUTPUT_DIR / 'runs'
print("Hasil akan disimpan di:", OUTPUT_DIR)

## 4. Menyiapkan dataset deteksi (format YOLO)

### 4.1 Format yang dibutuhkan
```
dataset/
├── data.yaml            # daftar kelas + lokasi split
├── train/
│   ├── images/  xxx.jpg
│   └── labels/  xxx.txt # satu baris per jerawat
├── valid/ (images, labels)
└── test/  (images, labels)
```
Setiap baris file label: `class_id x_center y_center width height` — semua koordinat **dinormalisasi 0–1** terhadap ukuran gambar.
Contoh: `1 0.412 0.530 0.031 0.028` artinya satu jerawat kelas 1 (mis. Papula) di sekitar tengah gambar dengan ukuran ±3% lebar gambar.

**Jumlah baris pada file label = jumlah jerawat sebenarnya (ground truth)** — inilah yang nanti dibandingkan dengan hasil hitungan model.

### 4.2 Cara membuat anotasi dari dataset Anda (disarankan)
1. Buat project baru di [roboflow.com](https://roboflow.com) dengan tipe **Object Detection**.
2. Upload gambar dari folder `train/`, `valid/`, `test/` dataset Anda.
3. Gambar kotak pada **SETIAP** jerawat yang terlihat. Kotak dibuat rapat (*tight*) di sekitar lesi.
   Jerawat yang tidak dianotasi akan dianggap "latar belakang", sehingga model belajar untuk **tidak** menghitungnya — konsistensi anotasi adalah kunci akurasi counting.
4. Pilih skema kelas:
   - **3 kelas** (`Cyst`, `Papula`, `Pustula`) → bisa menghitung total **dan** per jenis; atau
   - **1 kelas** (`acne`) → lebih mudah dianotasi dan biasanya lebih akurat untuk total. Bagus sebagai baseline.
   Kode notebook ini otomatis menyesuaikan jumlah kelas.
5. **Generate Version** (split mis. 70/20/10). Augmentasi di Roboflow tidak wajib karena Ultralytics sudah melakukan augmentasi saat training.
6. **Export Dataset → format YOLOv8** (format teks yang sama dipakai YOLO11/YOLO26) → salin *workspace*, *project*, dan *version*.

### 4.3 Menyimpan API key dengan aman
Jangan menulis API key langsung di notebook. Di Colab klik ikon **🔑 Secrets** (sidebar kiri) → tambahkan
`ROBOFLOW_API_KEY` → aktifkan *Notebook access*. Kode di bawah membacanya lewat `userdata.get(...)`.

### 4.4 Alternatif: file ZIP
Jika dataset sudah dalam bentuk ZIP berformat YOLO (misalnya hasil export manual), letakkan di Google Drive dan pilih `DATA_SOURCE = "zip"`.

In [ ]:
import shutil

DATA_SOURCE = "roboflow"     # "roboflow" atau "zip"
DATASET_DIR = Path('/content/datasets/acne_det')

if DATA_SOURCE == "roboflow":
    from google.colab import userdata
    from roboflow import Roboflow

    RF_WORKSPACE = "nama-workspace-anda"   # <-- ganti
    RF_PROJECT   = "nama-project-anda"     # <-- ganti
    RF_VERSION   = 1                       # <-- ganti

    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
        "yolov8", location=str(DATASET_DIR), overwrite=True)

elif DATA_SOURCE == "zip":
    ZIP_PATH = "/content/drive/MyDrive/dataset_jerawat_yolo.zip"   # <-- ganti
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.unpack_archive(ZIP_PATH, DATASET_DIR)

else:
    raise ValueError("DATA_SOURCE harus 'roboflow' atau 'zip'")

print("Isi folder dataset:")
for p in sorted(DATASET_DIR.rglob('*')):
    if p.is_dir() and len(p.relative_to(DATASET_DIR).parts) <= 3:
        print("  ", p.relative_to(DATASET_DIR))

## 5. Merapikan `data.yaml`
File `data.yaml` dari Roboflow sering berisi path relatif seperti `../train/images` yang bisa salah ketika folder dipindah.
Sel ini:
1. Mencari `data.yaml` di dalam folder dataset.
2. Mendeteksi folder `train/valid/test` yang benar-benar ada.
3. Menulis ulang file baru `data_fixed.yaml` dengan **path absolut**.
4. Menyimpan daftar nama kelas ke variabel `CLASS_NAMES` untuk dipakai di seluruh notebook.

In [ ]:
import yaml

DATA_YAML = next(DATASET_DIR.rglob('data.yaml'))
DATA_ROOT = DATA_YAML.parent
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

def find_split(*candidates):
    for name in candidates:
        p = DATA_ROOT / name / 'images'
        if p.exists():
            return str(p)
    return None

cfg.pop('path', None)
cfg['train'] = find_split('train')
cfg['val']   = find_split('valid', 'val')
cfg['test']  = find_split('test')
if cfg['train'] is None or cfg['val'] is None:
    raise FileNotFoundError("Folder train/images atau valid/images tidak ditemukan. Periksa struktur dataset.")
if cfg['test'] is None:
    cfg.pop('test')
    print("Peringatan: split test tidak ada, evaluasi akhir akan memakai split validasi.")

names = cfg['names']
CLASS_NAMES = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
cfg['names'] = CLASS_NAMES
cfg['nc'] = len(CLASS_NAMES)

FIXED_YAML = DATA_ROOT / 'data_fixed.yaml'
with open(FIXED_YAML, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

EVAL_SPLIT = 'test' if 'test' in cfg else 'val'
print(open(FIXED_YAML).read())
print("Kelas:", CLASS_NAMES)

## 6. Eksplorasi dataset (EDA)
Sebelum training, pahami data Anda. Informasi ini juga penting untuk bab metodologi:
- **Jumlah gambar & jumlah jerawat (instances)** per split dan per kelas → apakah kelas tidak seimbang?
  (Pada dataset klasifikasi Anda, `Papula` jauh lebih banyak daripada `Pustula`.)
- **Distribusi jumlah jerawat per gambar** → rentang nilai yang harus bisa dihitung model.
- **Ukuran bounding box** → jerawat biasanya < 32×32 piksel (kategori *small object*), alasan memakai `IMGSZ` yang cukup besar.

Fungsi `read_labels()` membaca file label YOLO dan mengubahnya menjadi koordinat piksel `x1,y1,x2,y2`.
Jika label ternyata berbentuk poligon (segmentasi), poligon otomatis dikonversi menjadi kotak.

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from collections import Counter

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(split_dir):
    return sorted(p for p in Path(split_dir).iterdir() if p.suffix.lower() in IMG_EXTS)

def label_path_for(img_path):
    img_path = Path(img_path)
    return img_path.parent.parent / 'labels' / (img_path.stem + '.txt')

def read_labels(img_path, img_w=1, img_h=1):
    """Return (cls_ids, boxes_xyxy) dalam piksel dari file label YOLO."""
    lp = label_path_for(img_path)
    cls_ids, boxes = [], []
    if lp.exists():
        for line in lp.read_text().splitlines():
            vals = line.split()
            if len(vals) < 5:
                continue
            c, coords = int(float(vals[0])), np.array(vals[1:], dtype=float)
            if len(coords) == 4:                      # format bbox: xc yc w h
                xc, yc, w, h = coords
                x1, y1, x2, y2 = xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2
            else:                                     # format poligon: x1 y1 x2 y2 ...
                xs, ys = coords[0::2], coords[1::2]
                x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
            cls_ids.append(c)
            boxes.append([x1 * img_w, y1 * img_h, x2 * img_w, y2 * img_h])
    return np.array(cls_ids, dtype=int), np.array(boxes, dtype=float).reshape(-1, 4)

SPLITS = {s: cfg[s] for s in ['train', 'val', 'test'] if s in cfg}

rows, box_rows = [], []
for split, d in SPLITS.items():
    for img_path in list_images(d):
        cls_ids, boxes = read_labels(img_path, IMGSZ, IMGSZ)   # skala ke ukuran input model
        row = {'split': split, 'image': img_path.name, 'total': len(cls_ids)}
        row.update({n: int((cls_ids == i).sum()) for i, n in enumerate(CLASS_NAMES)})
        rows.append(row)
        for c, b in zip(cls_ids, boxes):
            box_rows.append({'split': split, 'class': CLASS_NAMES[c],
                             'w_px': b[2] - b[0], 'h_px': b[3] - b[1]})

df_gt = pd.DataFrame(rows)
df_box = pd.DataFrame(box_rows)

summary = df_gt.groupby('split').agg(
    gambar=('image', 'count'), total_jerawat=('total', 'sum'),
    rata2_per_gambar=('total', 'mean'), median=('total', 'median'),
    maks=('total', 'max'), gambar_tanpa_jerawat=('total', lambda x: int((x == 0).sum())))
print("Ringkasan per split:"); display(summary.round(2))
print("Jumlah jerawat per kelas:"); display(df_gt.groupby('split')[CLASS_NAMES].sum())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df_gt[df_gt.split == 'train']['total'].plot.hist(bins=30, ax=ax[0], color='tab:red')
ax[0].set(title='Distribusi jumlah jerawat per gambar (train)', xlabel='Jumlah jerawat', ylabel='Jumlah gambar')
if len(df_box):
    side = np.sqrt(df_box['w_px'] * df_box['h_px'])
    side.plot.hist(bins=40, ax=ax[1], color='tab:blue')
    ax[1].axvline(32, ls='--', c='k', label='32 px (batas small object COCO)')
    ax[1].set(title=f'Ukuran bbox (sqrt(w*h)) pada input {IMGSZ}px', xlabel='piksel'); ax[1].legend()
    print(f"Median ukuran bbox: {side.median():.1f} px | {(side < 32).mean()*100:.1f}% termasuk small object")
plt.tight_layout(); plt.show()

### 6.1 Visualisasi ground truth
Fungsi `draw_boxes()` dipakai untuk **ground truth maupun prediksi**, sehingga tampilannya konsisten:
kotak tipis berwarna per kelas + panel ringkasan **"Total jerawat"** dan jumlah per kelas di pojok kiri atas.
Periksa beberapa gambar untuk memastikan anotasi sudah benar sebelum training — kesalahan anotasi akan langsung menjadi kesalahan hitung.

In [ ]:
PALETTE = [(255, 56, 56), (56, 120, 255), (40, 200, 70), (255, 160, 0), (200, 0, 200), (0, 200, 200)]  # RGB

def draw_boxes(img_rgb, boxes_xyxy, cls_ids, confs=None, header=None):
    out = np.ascontiguousarray(img_rgb.copy())
    h, w = out.shape[:2]
    lw = max(1, round((h + w) / 2 * 0.003))
    for i, (b, c) in enumerate(zip(boxes_xyxy, cls_ids)):
        x1, y1, x2, y2 = map(int, b)
        color = PALETTE[int(c) % len(PALETTE)]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, lw)
        if confs is not None:
            cv2.putText(out, f"{confs[i]:.2f}", (x1, max(y1 - 2, 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3 * lw, color, 1, cv2.LINE_AA)
    counts = Counter(int(c) for c in cls_ids)
    lines = ([header] if header else []) + [f"Total jerawat: {len(cls_ids)}"]
    colors = [(255, 255, 255)] * len(lines)
    for k, n in enumerate(CLASS_NAMES):
        lines.append(f"{n}: {counts.get(k, 0)}")
        colors.append(PALETTE[k % len(PALETTE)])
    fs, th = 0.5, 20
    cv2.rectangle(out, (0, 0), (220, th * len(lines) + 8), (0, 0, 0), -1)
    for j, (t, col) in enumerate(zip(lines, colors)):
        cv2.putText(out, t, (6, th * (j + 1)), cv2.FONT_HERSHEY_SIMPLEX, fs, col, 1, cv2.LINE_AA)
    return out

def load_rgb(path):
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)

sample = list_images(cfg['train'])[:4]
fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5))
for ax, p in zip(np.atleast_1d(axes), sample):
    img = load_rgb(p)
    cls_ids, boxes = read_labels(p, img.shape[1], img.shape[0])
    ax.imshow(draw_boxes(img, boxes, cls_ids, header='Ground truth')); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Training YOLO26
Kita memakai **transfer learning**: model mulai dari bobot yang sudah dilatih pada dataset COCO (80 objek umum),
lalu disesuaikan (*fine-tuning*) untuk mengenali jerawat. Ini jauh lebih baik daripada training dari nol bila data sedikit.

Penjelasan argumen `model.train()`:
| Argumen | Arti |
|---|---|
| `data` | File `data_fixed.yaml` dari Langkah 5 |
| `epochs`, `imgsz`, `batch`, `patience`, `seed` | Dari konfigurasi Langkah 3 |
| `optimizer='auto'` | Ultralytics memilih optimizer & learning rate otomatis (untuk YOLO26 biasanya MuSGD/AdamW tergantung panjang training) |
| `cos_lr=True` | Learning rate turun mengikuti kurva cosinus → konvergensi lebih halus |
| `mosaic=1.0`, `close_mosaic=10` | Augmentasi *mosaic* (4 gambar digabung) membantu objek kecil; dimatikan pada 10 epoch terakhir agar model menyesuaikan dengan gambar asli |
| `fliplr=0.5` | Flip horizontal 50% — wajah kiri/kanan setara |
| `flipud=0.0` | Tidak flip vertikal — wajah terbalik tidak realistis |
| `hsv_h/s/v` | Variasi warna & kecerahan → model tahan terhadap perbedaan pencahayaan dan warna kulit |
| `cache=True` | Gambar disimpan di RAM → training lebih cepat |
| `project`, `name` | Lokasi penyimpanan hasil (di Google Drive bila `USE_DRIVE=True`) |

Selama training, perhatikan kolom `box_loss`, `cls_loss` (harus turun) dan `mAP50`, `mAP50-95` validasi (harus naik).
Bobot terbaik (berdasarkan *fitness* = kombinasi mAP) otomatis disimpan sebagai `weights/best.pt`.

> Jika sesi Colab terputus, jalankan ulang Langkah 1–5 lalu lanjutkan training dengan
> `YOLO(RUNS_DIR/'yolo26_acne/weights/last.pt').train(resume=True)`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_WEIGHTS)
train_results = model.train(
    data=str(FIXED_YAML),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED, deterministic=True,
    optimizer='auto', cos_lr=True,
    mosaic=1.0, close_mosaic=10, fliplr=0.5, flipud=0.0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    cache=True, workers=2, plots=True,
    project=str(RUNS_DIR), name='yolo26_acne', exist_ok=True,
)

TRAIN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = TRAIN_DIR / 'weights' / 'best.pt'
print("Bobot terbaik:", BEST_WEIGHTS)

## 8. Kurva training
Ultralytics otomatis membuat grafik di folder hasil training:
- **`results.png`** — loss train/val dan metrik per epoch. Jika loss validasi naik sementara loss train terus turun → indikasi *overfitting*.
- **`confusion_matrix_normalized.png`** — kelas mana yang sering tertukar (mis. Papula vs Pustula) dan berapa jerawat yang terlewat (kolom/baris *background*).
- **`BoxPR_curve.png`** — kurva Precision-Recall; luas di bawah kurva = AP.
- **`BoxF1_curve.png`** — F1 terhadap confidence; puncaknya memberi petunjuk ambang confidence yang baik.

In [ ]:
from IPython.display import Image as IPImage, display

for name in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png',
             'PR_curve.png', 'F1_curve.png']:
    f = TRAIN_DIR / name
    if f.exists():
        print(name); display(IPImage(filename=str(f), width=900))

## 9. Evaluasi deteksi pada data test
Model terbaik dievaluasi pada split **test** (data yang tidak pernah dilihat saat training maupun pemilihan model).

| Metrik | Arti dalam konteks jerawat |
|---|---|
| **Precision (P)** | Dari semua kotak yang diprediksi, berapa persen yang benar-benar jerawat. P rendah → model **menghitung lebih** (*overcount*). |
| **Recall (R)** | Dari semua jerawat sebenarnya, berapa persen yang ditemukan. R rendah → model **menghitung kurang** (*undercount*). |
| **mAP50** | Rata-rata AP semua kelas dengan syarat IoU ≥ 0.5 (kotak prediksi cukup tumpang-tindih dengan kotak asli). |
| **mAP50-95** | Rata-rata AP pada IoU 0.5 s/d 0.95 — lebih ketat terhadap ketepatan posisi kotak. |

`conf=0.001` dipakai khusus untuk menghitung mAP (standar evaluasi), bukan untuk counting.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(FIXED_YAML), split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH,
                         conf=0.001, plots=True, project=str(RUNS_DIR), name='eval_' + EVAL_SPLIT, exist_ok=True)

print(f"Split evaluasi : {EVAL_SPLIT}")
print(f"Precision      : {metrics.box.mp:.4f}")
print(f"Recall         : {metrics.box.mr:.4f}")
print(f"mAP50          : {metrics.box.map50:.4f}")
print(f"mAP50-95       : {metrics.box.map:.4f}")

per_class = []
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    per_class.append({'kelas': CLASS_NAMES[int(c)], 'precision': p, 'recall': r, 'mAP50': ap50, 'mAP50-95': ap})
df_det_metrics = pd.DataFrame(per_class).round(4)
display(df_det_metrics)

## 10. Fungsi penghitung jerawat
Prinsip counting berbasis deteksi sangat sederhana:
> **Jumlah jerawat = jumlah bounding box yang confidence-nya ≥ ambang (threshold).**

Karena YOLO26 bersifat **NMS-free**, setiap jerawat idealnya menghasilkan tepat satu kotak, sehingga tidak ada langkah pembersihan kotak ganda tambahan.

Dua fungsi disiapkan:
1. **`predict_raw()`** — menjalankan model **sekali** dengan `conf` sangat rendah dan menyimpan semua kotak beserta skornya.
   Dengan begitu kita bisa mencoba banyak threshold (Langkah 12) tanpa menjalankan model berulang kali.
2. **`count_from_raw()`** — menyaring kotak berdasarkan threshold, lalu menghitung **total** dan **per kelas**.

Fungsi **`count_acne()`** adalah versi "siap pakai" untuk satu gambar: mengembalikan jumlah + gambar beranotasi.

In [ ]:
def predict_raw(model, image_paths, imgsz=IMGSZ, min_conf=0.01):
    """Jalankan model sekali; simpan semua kotak (xyxy, cls, conf) per gambar."""
    raw = {}
    for r in model.predict([str(p) for p in image_paths], imgsz=imgsz, conf=min_conf,
                           max_det=300, stream=True, verbose=False):
        b = r.boxes
        raw[Path(r.path).name] = {
            'xyxy': b.xyxy.cpu().numpy(),
            'cls':  b.cls.cpu().numpy().astype(int),
            'conf': b.conf.cpu().numpy(),
        }
    return raw

def count_from_raw(det, conf_thr):
    keep = det['conf'] >= conf_thr
    cls = det['cls'][keep]
    counts = {'total': int(keep.sum())}
    counts.update({n: int((cls == i).sum()) for i, n in enumerate(CLASS_NAMES)})
    return counts, keep

def count_acne(model, image_path, conf_thr=CONF_DEFAULT, imgsz=IMGSZ):
    """Hitung jerawat pada satu gambar. Return: (dict jumlah, gambar RGB beranotasi)."""
    det = predict_raw(model, [image_path], imgsz=imgsz, min_conf=conf_thr)[Path(image_path).name]
    counts, keep = count_from_raw(det, conf_thr)
    vis = draw_boxes(load_rgb(image_path), det['xyxy'][keep], det['cls'][keep],
                     confs=det['conf'][keep], header=f'Prediksi (conf>={conf_thr:.2f})')
    return counts, vis

# Uji cepat pada satu gambar test
test_images = list_images(cfg[EVAL_SPLIT])
counts, vis = count_acne(best_model, test_images[0])
print(counts)
plt.figure(figsize=(7, 7)); plt.imshow(vis); plt.axis('off'); plt.show()

## 11. Demo visual: ground truth vs prediksi
Menampilkan beberapa gambar test berdampingan. Kiri = anotasi manusia (jumlah sebenarnya), kanan = hasil model.
Perhatikan pola kesalahan: apakah model melewatkan jerawat kecil/pucat (*false negative*),
atau salah mendeteksi pori-pori, tahi lalat, bayangan, dan bekas jerawat (*false positive*)?
Pengamatan kualitatif ini berguna untuk bagian pembahasan penelitian.

In [ ]:
N_SHOW = 4
fig, axes = plt.subplots(N_SHOW, 2, figsize=(12, 6 * N_SHOW))
for row, p in zip(axes, test_images[:N_SHOW]):
    img = load_rgb(p)
    gt_cls, gt_boxes = read_labels(p, img.shape[1], img.shape[0])
    row[0].imshow(draw_boxes(img, gt_boxes, gt_cls, header='Ground truth'))
    _, vis = count_acne(best_model, p)
    row[1].imshow(vis)
    for a in row: a.axis('off')
plt.tight_layout(); plt.show()

## 12. Tuning confidence threshold untuk counting (pakai data validasi)
mAP mengukur kualitas deteksi di **semua** threshold, tetapi untuk counting kita harus memilih **satu** threshold.
- Threshold terlalu rendah → banyak kotak palsu → *overcount*.
- Threshold terlalu tinggi → jerawat samar terbuang → *undercount*.

Kita mencari threshold yang memberi **MAE (Mean Absolute Error) terkecil** pada split **validasi**.
Data test **tidak** dipakai di sini agar hasil akhir tetap jujur (tidak ada *data leakage*).

Metrik counting yang dipakai:
| Metrik | Rumus / arti |
|---|---|
| **MAE** | rata-rata \|prediksi − sebenarnya\| → "rata-rata meleset berapa jerawat per gambar" |
| **RMSE** | akar rata-rata kuadrat error → menghukum kesalahan besar lebih berat |
| **Bias (ME)** | rata-rata (prediksi − sebenarnya); positif = cenderung overcount, negatif = undercount |
| **rMAE** | Σ\|error\| / Σ jumlah sebenarnya → error relatif terhadap total jerawat |
| **Acc exact / ±1 / ±2** | persentase gambar yang hitungannya tepat / meleset maksimal 1 / maksimal 2 |
| **Pearson r** | korelasi antara hitungan model dan hitungan manusia |

In [ ]:
def gt_counts(image_paths):
    out = {}
    for p in image_paths:
        cls_ids, _ = read_labels(p)
        c = {'total': len(cls_ids)}
        c.update({n: int((cls_ids == i).sum()) for i, n in enumerate(CLASS_NAMES)})
        out[Path(p).name] = c
    return out

def counting_metrics(gt, pred):
    gt, pred = np.asarray(gt, float), np.asarray(pred, float)
    err = pred - gt
    r = np.corrcoef(gt, pred)[0, 1] if gt.std() > 0 and pred.std() > 0 else np.nan
    return {
        'MAE': np.abs(err).mean(),
        'RMSE': np.sqrt((err ** 2).mean()),
        'Bias': err.mean(),
        'rMAE': np.abs(err).sum() / max(gt.sum(), 1),
        'Acc_exact': (err == 0).mean(),
        'Acc_pm1': (np.abs(err) <= 1).mean(),
        'Acc_pm2': (np.abs(err) <= 2).mean(),
        'Pearson_r': r,
    }

val_images = list_images(cfg['val'])
raw_val = predict_raw(best_model, val_images)
gt_val = gt_counts(val_images)
names_val = list(gt_val)

thresholds = np.round(np.arange(0.05, 0.91, 0.05), 2)
sweep = []
for t in thresholds:
    pred = [count_from_raw(raw_val[n], t)[0]['total'] for n in names_val]
    gt = [gt_val[n]['total'] for n in names_val]
    sweep.append({'conf': t, **counting_metrics(gt, pred)})
df_sweep = pd.DataFrame(sweep)

BEST_CONF = float(df_sweep.loc[df_sweep['MAE'].idxmin(), 'conf'])
print(f"Threshold terbaik (MAE minimum di validasi): {BEST_CONF}")
display(df_sweep.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_sweep['conf'], df_sweep['MAE'], 'o-', label='MAE')
ax.plot(df_sweep['conf'], df_sweep['Bias'], 's--', label='Bias (pred - GT)')
ax.axvline(BEST_CONF, c='r', ls=':', label=f'terbaik = {BEST_CONF}')
ax.axhline(0, c='gray', lw=0.8)
ax.set(xlabel='Confidence threshold', ylabel='Jerawat per gambar', title='Pengaruh threshold terhadap counting (validasi)')
ax.legend(); plt.show()

## 13. Evaluasi counting pada data test
Dengan `BEST_CONF` dari validasi, kita mengukur kemampuan counting secara final di data test:
1. Tabel per gambar: jumlah sebenarnya vs prediksi (total & per kelas).
2. Metrik counting total dan per kelas.
3. **Scatter plot** prediksi vs sebenarnya — titik pada garis diagonal = hitungan tepat.
   Titik di atas garis = overcount, di bawah garis = undercount.
   Biasanya error membesar pada gambar dengan jerawat sangat banyak (lesi saling berdekatan/bertumpuk).

Laporkan **dua jenis hasil** di penelitian: metrik deteksi (Langkah 9) **dan** metrik counting (langkah ini).

In [ ]:
raw_test = predict_raw(best_model, test_images)
gt_test = gt_counts(test_images)

table = []
for n in gt_test:
    pc, _ = count_from_raw(raw_test[n], BEST_CONF)
    row = {'image': n}
    for k in ['total'] + CLASS_NAMES:
        row[f'gt_{k}'] = gt_test[n][k]
        row[f'pred_{k}'] = pc[k]
    row['error_total'] = row['pred_total'] - row['gt_total']
    table.append(row)
df_count = pd.DataFrame(table)

metric_rows = []
for k in ['total'] + CLASS_NAMES:
    metric_rows.append({'target': k, **counting_metrics(df_count[f'gt_{k}'], df_count[f'pred_{k}'])})
df_count_metrics = pd.DataFrame(metric_rows).set_index('target').round(3)

print(f"Evaluasi counting pada split '{EVAL_SPLIT}' dengan conf = {BEST_CONF}")
display(df_count_metrics)
print("Gambar dengan error terbesar:")
display(df_count.reindex(df_count['error_total'].abs().sort_values(ascending=False).index).head(10))

mx = max(df_count['gt_total'].max(), df_count['pred_total'].max()) + 2
plt.figure(figsize=(6, 6))
plt.scatter(df_count['gt_total'], df_count['pred_total'], alpha=0.7)
plt.plot([0, mx], [0, mx], 'r--', label='prediksi = sebenarnya')
plt.xlabel('Jumlah jerawat sebenarnya (GT)'); plt.ylabel('Jumlah jerawat prediksi')
plt.title(f"Counting test — MAE={df_count_metrics.loc['total','MAE']:.2f}, r={df_count_metrics.loc['total','Pearson_r']:.2f}")
plt.legend(); plt.axis('equal'); plt.xlim(0, mx); plt.ylim(0, mx); plt.show()

## 14. Hitung jerawat pada gambar baru
### 14.1 Upload satu/beberapa gambar
Jalankan sel di bawah, pilih gambar wajah dari komputer Anda, dan model akan menampilkan kotak + jumlah jerawat.
Gambar beranotasi disimpan ke `OUTPUT_DIR/hasil_prediksi/`.

Tips agar hasil akurat: gunakan foto dengan pencahayaan merata, fokus tajam, dan jarak/sudut yang mirip dengan data training.

In [ ]:
from google.colab import files

PRED_DIR = OUTPUT_DIR / 'hasil_prediksi'
PRED_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for fname in uploaded:
    counts, vis = count_acne(best_model, fname, conf_thr=BEST_CONF)
    print(fname, '->', counts)
    cv2.imwrite(str(PRED_DIR / f"pred_{Path(fname).stem}.jpg"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    plt.figure(figsize=(7, 7)); plt.imshow(vis); plt.axis('off'); plt.title(fname); plt.show()

### 14.2 Hitung semua gambar dalam satu folder → CSV
Berguna untuk menghitung jerawat pada banyak gambar sekaligus, misalnya **seluruh gambar dataset klasifikasi asli Anda**
(`Acne Classification.v12i.folder`) yang diunggah ke Google Drive. Folder dibaca secara rekursif, dan nama subfolder
(Cyst/Papula/Pustula) ikut dicatat sebagai label klasifikasi asli — sehingga Anda bisa menganalisis, misalnya,
rata-rata jumlah jerawat pada gambar berlabel "Cyst" vs "Papula".

In [ ]:
INFER_DIR = Path('/content/drive/MyDrive/Acne Classification.v12i.folder')   # <-- ganti sesuai lokasi folder

if INFER_DIR.exists():
    infer_images = sorted(p for p in INFER_DIR.rglob('*') if p.suffix.lower() in IMG_EXTS)
    raw_inf = {}
    for p in infer_images:   # per gambar agar nama file kembar di subfolder berbeda tidak saling menimpa
        raw_inf[str(p)] = predict_raw(best_model, [p])[p.name]
    rows = []
    for p in infer_images:
        c, _ = count_from_raw(raw_inf[str(p)], BEST_CONF)
        rows.append({'path': str(p.relative_to(INFER_DIR)), 'label_folder': p.parent.name, **c})
    df_infer = pd.DataFrame(rows)
    df_infer.to_csv(OUTPUT_DIR / 'hitung_jerawat_folder.csv', index=False)
    print(f"{len(df_infer)} gambar diproses. CSV disimpan di {OUTPUT_DIR / 'hitung_jerawat_folder.csv'}")
    display(df_infer.head())
    display(df_infer.groupby('label_folder')['total'].describe().round(2))
else:
    print("Folder tidak ditemukan, lewati langkah ini:", INFER_DIR)

## 15. Simpan hasil eksperimen
Menyimpan semua artefak penting ke `OUTPUT_DIR` (Google Drive):
- `best_yolo26_acne.pt` — bobot model terbaik (bisa dimuat lagi dengan `YOLO('best_yolo26_acne.pt')`).
- `metrik_deteksi.csv`, `metrik_counting.csv`, `counting_per_gambar.csv`, `sweep_threshold_val.csv` — untuk tabel di laporan.
- `config_eksperimen.yaml` — catatan parameter eksperimen agar dapat direproduksi.

Opsional: ekspor ke **ONNX** untuk dipakai di aplikasi (YOLO26 tanpa NMS sehingga ekspor lebih sederhana).

In [ ]:
shutil.copy(BEST_WEIGHTS, OUTPUT_DIR / 'best_yolo26_acne.pt')
df_det_metrics.to_csv(OUTPUT_DIR / 'metrik_deteksi.csv', index=False)
df_count_metrics.to_csv(OUTPUT_DIR / 'metrik_counting.csv')
df_count.to_csv(OUTPUT_DIR / 'counting_per_gambar.csv', index=False)
df_sweep.to_csv(OUTPUT_DIR / 'sweep_threshold_val.csv', index=False)

experiment = {
    'model': MODEL_WEIGHTS, 'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'patience': PATIENCE,
    'seed': SEED, 'classes': CLASS_NAMES, 'best_conf': BEST_CONF, 'eval_split': EVAL_SPLIT,
    'ultralytics_version': ultralytics.__version__,
    'det_mAP50': float(metrics.box.map50), 'det_mAP50_95': float(metrics.box.map),
    'count_MAE': float(df_count_metrics.loc['total', 'MAE']),
    'count_RMSE': float(df_count_metrics.loc['total', 'RMSE']),
}
with open(OUTPUT_DIR / 'config_eksperimen.yaml', 'w') as f:
    yaml.safe_dump(experiment, f, sort_keys=False)
print(yaml.safe_dump(experiment, sort_keys=False))

# best_model.export(format='onnx', imgsz=IMGSZ)   # opsional

## Langkah berikutnya untuk penelitian
1. **Baseline perbandingan**: latih `yolo26n` vs `yolo26s` vs `yolo26m`, dan bandingkan dengan YOLO11 / YOLOv8 (cukup ganti `MODEL_WEIGHTS`, mis. `yolo11s.pt`) pada split yang sama.
2. **1 kelas vs 3 kelas**: apakah menggabungkan semua jenis menjadi satu kelas `acne` menurunkan MAE total?
3. **Resolusi input**: coba `IMGSZ = 800/1024` — sering membantu objek kecil, dengan biaya waktu komputasi.
4. **Tiling / SAHI**: memotong gambar besar menjadi petak-petak kecil untuk jerawat yang sangat kecil.
5. **Analisis kesalahan**: kelompokkan error berdasarkan jumlah jerawat (sedikit/sedang/banyak) dan jenis kulit/pencahayaan.
6. **Severity grading**: gunakan jumlah jerawat hasil model untuk menentukan tingkat keparahan (mis. skala Hayashi: ringan 0–5, sedang 6–20, berat 21–50, sangat berat > 50 lesi per sisi wajah).
7. **Validasi silang (k-fold)** bila dataset kecil, agar hasil lebih stabil secara statistik.